# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors with `mlcroissant`
This notebook demonstrates step-by-step how to explore the [FAIR^2 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library for easy, standards-based data access and processing.

### Dataset Source
The dataset is accessed via its [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for self-describing data loading and structure.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

# Define the Croissant metadata URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
# Metadata object (not a dict)
metadata = dataset.metadata
print(f"Dataset Name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. This will help identify which parts of the data to load.

In [ ]:
# List available record sets and their IDs
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets or len(record_sets) == 0:
    # Try loading record set IDs from the dataset instance
    # This approach works for mlcroissant>=0.6.1
    # Seek record_set_ids via dataset._metadata._record_sets if necessary
    try:
        record_sets = [rs['@id'] for rs in dataset._metadata._record_sets]
    except Exception:
        record_sets = []

print('Record sets in this dataset:')
if len(record_sets):
    for rs_id in record_sets:
        print(f"- {rs_id}")
else:
    print('No explicit record sets in metadata; will attempt to load primary records.')

# Let's try to show sample fields for each record set (if possible):
print("\nAvailable fields (columns) in the main record set (by @id):")
first_record_set = None
if len(record_sets):
    first_record_set = record_sets[0]
    sample = next(dataset.records(record_set=first_record_set), None)
else:
    # Try without specifying record_set (the default/main set)
    sample = next(dataset.records(), None)

if sample:
    for field in sample.keys():
        print(f"- {field}")
else:
    print("No records available for preview.")

## 3. Data Extraction
Load data from the main record set using its `@id`. We use the record set and field `@id`s found in the previous step for clean referencing.

In [ ]:
# Choose which record set(s) to extract based on the previous overview
# If record_sets is empty, we'll try loading the default/primary record set (most Croissant datasets have one)
if len(record_sets):
    record_set_ids = record_sets
else:
    record_set_ids = [None]

dataframes = {}
for record_set_id in record_set_ids:
    # Use None for default/first record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    # Use the @id or a string key
    key = record_set_id if record_set_id else 'default_set'
    dataframes[key] = df

chosen_set = record_set_ids[0] if record_set_ids[0] else 'default_set'
print(f"Loaded DataFrame columns for record set '{chosen_set}':")
print(dataframes[chosen_set].columns.tolist())

print("\nSample rows:")
display(dataframes[chosen_set].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the main dataset: filter by a numeric field (e.g., `Age_at_Second_CRC`), normalize it, and group by a relevant categorical variable (e.g., `Sex`).

> **Note:** Referencing all entities by their `@id`. Adjust field IDs as found above.

In [ ]:
# Find a numeric field for analysis
df = dataframes[chosen_set]

# List numeric-like columns by dtype if not known from schema
numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
if not numeric_cols:
    # Attempt with common names if not detected
    candidates = [col for col in df.columns if 'age' in col.lower() or 'year' in col.lower()]
    numeric_field = candidates[0] if candidates else df.columns[0]
else:
    numeric_field = numeric_cols[0]

print(f"Using numeric field: {numeric_field}")
threshold = 50
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize numeric field
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, norm_col]].head())

# Try grouping by a categorical variable, e.g., Sex
possible_group_fields = [col for col in df.columns if col.lower() in ['sex', 'gender', 'msi_status'] or 'sex' in col.lower()]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable grouping categorical field available.")

## 5. Visualization
Visualize the distribution of a selected numeric field and compare across groups, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Side-by-side distributions by group if group_field present
if group_field and group_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to use the `mlcroissant` library to load a FAIR^2-compliant colorectal cancer dataset, referencing all data elements by `@id`.
- We explored data fields, filtered and normalized a numeric variable, grouped by a clinical feature, and visualized distributions.
- The dataset is well-structured and supports rich exploration for clinical oncology analytics, particularly for MSI/CRC studies in cancer survivors.

For further analysis, you might explore additional fields or more advanced modeling/visualization. See the [Croissant specification](https://mlcommons.org/croissant) for more info.